In [27]:
# Climate Risk & Weather Agent
## Maharashtra

In [28]:
import requests
import pandas as pd

In [29]:
API_KEY = "97a227f1b6b34be26ac4b556d2d60e59"

In [30]:
def fetch_weather(lat, lon):

    url = f"https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={API_KEY}&units=metric"
    data = requests.get(url).json()

    return {
        "temp": data["main"]["temp"],
        "humidity": data["main"]["humidity"],
        "rainfall": data.get("rain", {}).get("1h", 0),
        "wind": data["wind"]["speed"]
    }


In [31]:
def heat_stress(temp):
    if temp > 35:
        return 1
    elif temp >= 30:
        return 0.5
    return 0


def drought_risk(rainfall):
    if rainfall < 2:
        return 1
    elif rainfall < 10:
        return 0.5
    return 0


def humidity_risk(humidity):
    if humidity > 85:
        return 1
    elif humidity > 70:
        return 0.5
    return 0


In [32]:
def weather_risk_score(weather):
    return round(
        0.4 * heat_stress(weather["temp"]) +
        0.4 * drought_risk(weather["rainfall"]) +
        0.2 * humidity_risk(weather["humidity"]),
        2
    )


In [33]:
crop_requirements = {
    "Cotton": (21, 40),
    "Wheat": (10, 25),
    "Soybean": (15, 35)
}

def crop_suitability(weather):
    temp = weather["temp"]
    suitability = {}

    for crop, (min_t, max_t) in crop_requirements.items():
        suitability[crop] = 0.8 if min_t <= temp <= max_t else 0.4

    return suitability


In [34]:
def generate_weather_alerts(weather):
    alerts = []
    temp = weather["temp"]
    rainfall = weather["rainfall"]

    if temp >= 38:
        alerts.append("Heatwave risk. Avoid sowing and irrigate adequately.")

    if temp <= 8:
        alerts.append("Cold stress risk. Protect crops from low temperature.")

    if rainfall >= 80:
        alerts.append("Heavy rainfall expected. Ensure proper drainage.")

    if rainfall <= 5:
        alerts.append("Low rainfall. Irrigation required.")

    if not alerts:
        alerts.append("Weather conditions are normal. No immediate risk.")

    return alerts


In [35]:
def get_coordinates(village):

    url = f"http://api.openweathermap.org/geo/1.0/direct?q={village},Maharashtra,IN&limit=1&appid={API_KEY}"

    data = requests.get(url).json()

    if len(data) == 0:
        return None, None

    lat = data[0]["lat"]
    lon = data[0]["lon"]

    return lat, lon

In [36]:
def fetch_forecast(lat, lon):

    url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API_KEY}&units=metric"
    
    data = requests.get(url).json()

    temps = []
    rainfall = []

    for entry in data["list"][:10]:   # next ~30 hours
        temps.append(entry["main"]["temp"])
        rainfall.append(entry.get("rain", {}).get("3h", 0))

    return {
        "avg_temp": sum(temps)/len(temps),
        "total_rainfall": sum(rainfall)
    }
    

In [37]:
def forecast_risk_score(forecast):

    temp_risk = heat_stress(forecast["avg_temp"])
    rain_risk = drought_risk(forecast["total_rainfall"])

    return round(0.6 * temp_risk + 0.4 * rain_risk, 2)

In [38]:
def risk_level(score):

    if score < 0.3:
        return "Low"

    elif score < 0.6:
        return "Moderate"

    else:
        return "High"

In [39]:
def farmer_advice(current_risk, future_risk):

    advice = []

    if current_risk == "High":
        advice.append("Current weather conditions may harm crops.")

    if future_risk == "High":
        advice.append("High risk expected in coming days. Plan irrigation and avoid sowing.")

    if current_risk == "Low" and future_risk == "Low":
        advice.append("Weather conditions are good for farming.")

    return advice

In [40]:
def climate_agent(lat=None, lon=None, village=None):

    # If GPS coordinates are given
    if lat is not None and lon is not None:
        pass

    # Otherwise use village name
    elif village is not None:
        lat, lon = get_coordinates(village)

        if lat is None:
            return {"error": "Village not found"}

    else:
        return {"error": "Location not provided"}

    weather = fetch_weather(lat, lon)
    forecast = fetch_forecast(lat, lon)

    current_score = weather_risk_score(weather)
    future_score = forecast_risk_score(forecast)

    current_level = risk_level(current_score)
    future_level = risk_level(future_score)

    advice = farmer_advice(current_level, future_level)

    return {
    "latitude": lat,
    "longitude": lon,
    "temperature": weather["temp"],
    "humidity": weather["humidity"],
    "rainfall": weather["rainfall"],

    "current_weather_risk": current_level,
    "future_weather_risk": future_level,

    "alerts": generate_weather_alerts(weather),
    "crop_suitability": crop_suitability(weather),
    "farmer_advice": advice
}

In [41]:
climate_agent(lat=18.5204, lon=73.8567)


{'latitude': 18.5204,
 'longitude': 73.8567,
 'temperature': 34.92,
 'humidity': 15,
 'rainfall': 0,
 'current_weather_risk': 'High',
 'future_weather_risk': 'Moderate',
 'alerts': ['Low rainfall. Irrigation required.'],
 'crop_suitability': {'Cotton': 0.8, 'Wheat': 0.4, 'Soybean': 0.8},
 'farmer_advice': ['Current weather conditions may harm crops.']}

In [42]:
climate_agent(village="Dighi")

{'latitude': 20.8639649,
 'longitude': 76.33947898667083,
 'temperature': 37.24,
 'humidity': 12,
 'rainfall': 0,
 'current_weather_risk': 'High',
 'future_weather_risk': 'High',
 'alerts': ['Low rainfall. Irrigation required.'],
 'crop_suitability': {'Cotton': 0.8, 'Wheat': 0.4, 'Soybean': 0.4},
 'farmer_advice': ['Current weather conditions may harm crops.',
  'High risk expected in coming days. Plan irrigation and avoid sowing.']}

### ADVISOR AGENT 
-- for now gives advice based on only weather agent. Need to integrate soil agent as well.

In [43]:
def advisor_agent(weather_output):

    suitability = weather_output["crop_suitability"]
    future_risk = weather_output["future_weather_risk"]

    # choose crop with highest suitability
    best_crop = max(suitability, key=suitability.get)

    advice = []

    if future_risk == "High":
        advice.append("Weather risk is high. Delay sowing if possible.")

    advice.append(f"Recommended crop based on climate conditions: {best_crop}")

    return {
        "recommended_crop": best_crop,
        "advisor_message": advice
    }

In [44]:
def agrovision_system(lat=None, lon=None, village=None):

    weather_output = climate_agent(lat=lat, lon=lon, village=village)

    advisor_output = advisor_agent(weather_output)

    final_output = {
        "weather_analysis": weather_output,
        "advisor_recommendation": advisor_output
    }

    return final_output

In [47]:
agrovision_system(village="Dighi")

{'weather_analysis': {'latitude': 20.8639649,
  'longitude': 76.33947898667083,
  'temperature': 37.24,
  'humidity': 12,
  'rainfall': 0,
  'current_weather_risk': 'High',
  'future_weather_risk': 'High',
  'alerts': ['Low rainfall. Irrigation required.'],
  'crop_suitability': {'Cotton': 0.8, 'Wheat': 0.4, 'Soybean': 0.4},
  'farmer_advice': ['Current weather conditions may harm crops.',
   'High risk expected in coming days. Plan irrigation and avoid sowing.']},
 'advisor_recommendation': {'recommended_crop': 'Cotton',
  'advisor_message': ['Weather risk is high. Delay sowing if possible.',
   'Recommended crop based on climate conditions: Cotton']}}